# YouTube retriever batch export

Runs every retrieval function from `youtube_prod_comment_summary_embeddings.ipynb` over a list of **themes** and writes **one CSV per function** into an output folder.

Does **not** replace the manual *Run test queries* cells in the prod notebook.

## Output format

- **CSV** (default): one file per function, e.g. `top_creators.csv`. All themes are rows in the same file with a `theme` column. Nested fields (`matches`, `sample_videos`, …) are stored as **JSON strings** in cells — easy to open in Excel/Sheets and fine for analytics.
- **JSONL** would be better only if you need deeply nested documents without flattening; say if you want that variant later.

## Prerequisites

1. `.env` with Neo4j + embedding credentials (same as prod notebook).
2. Prod graph populated and vector indexes built.
3. Run **Setup** then **Load retrievers** below (loads `query-helpers` from the prod notebook).

In [ ]:
import csv
import json
import os
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, Iterator, List, Optional, Tuple

from dotenv import load_dotenv
from neo4j import GraphDatabase
from openai import AzureOpenAI, OpenAI

load_dotenv()

# --- Batch config (edit here) ------------------------------------------------
TOPICS: List[str] = [
    'gambling',
    'mental health',
    'beauty',
]

TOP_N = int(os.getenv('YT_RETRIEVER_TOP_N', '10'))
MIN_SCORE = float(os.getenv('YT_RETRIEVER_MIN_SCORE', '0.75'))
K_COUNT = int(os.getenv('YT_RETRIEVER_K_COUNT', '20000'))

_cwd = Path.cwd()
if (_cwd / 'youtube_prod_comment_summary_embeddings.ipynb').exists():
    NOTEBOOK_DIR = _cwd
elif (_cwd / 'notebooks' / 'youtube_prod_comment_summary_embeddings.ipynb').exists():
    NOTEBOOK_DIR = _cwd / 'notebooks'
else:
    NOTEBOOK_DIR = _cwd
PROD_NOTEBOOK = NOTEBOOK_DIR / 'youtube_prod_comment_summary_embeddings.ipynb'
OUTPUT_DIR = Path(os.getenv(
    'YT_RETRIEVER_EXPORT_DIR',
    str(NOTEBOOK_DIR / 'retriever_exports' / 'youtube'),
))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Topics:', TOPICS)
print('TOP_N=%s  MIN_SCORE=%s  K_COUNT=%s' % (TOP_N, MIN_SCORE, K_COUNT))
print('Output dir:', OUTPUT_DIR.resolve())
print('Prod notebook:', PROD_NOTEBOOK.resolve())

In [ ]:
# --- Minimal prod setup (Neo4j + embeddings) ---------------------------------

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USER = os.getenv('NEO4J_USER')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

AZURE_OPENAI_EMBEDDING_ENDPOINT = os.getenv('AZURE_OPENAI_EMBEDDING_ENDPOINT')
AZURE_OPENAI_EMBEDDING_API_KEY = os.getenv('AZURE_OPENAI_EMBEDDING_API_KEY')
AZURE_OPENAI_EMBEDDING_API_VERSION = os.getenv('AZURE_OPENAI_EMBEDDING_API_VERSION', '2024-02-01')
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = (
    os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT_MODEL_NAME')
    or os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME')
    or os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
    or 'text-embedding-3-large'
)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_EMBEDDING_MODEL = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-large')
EMBED_BATCH = int(os.getenv('YT_SUMMARY_EMBED_BATCH', '128'))
YOUTUBE_PLATFORM = 'youtube'

assert NEO4J_URI and NEO4J_USER and NEO4J_PASSWORD, 'Set NEO4J_* in .env'


def build_embedding_client():
    if AZURE_OPENAI_EMBEDDING_ENDPOINT and AZURE_OPENAI_EMBEDDING_API_KEY:
        client = AzureOpenAI(
            azure_endpoint=AZURE_OPENAI_EMBEDDING_ENDPOINT,
            api_key=AZURE_OPENAI_EMBEDDING_API_KEY,
            api_version=AZURE_OPENAI_EMBEDDING_API_VERSION,
        )
        return client, AZURE_OPENAI_EMBEDDING_DEPLOYMENT
    if OPENAI_API_KEY:
        return OpenAI(api_key=OPENAI_API_KEY), OPENAI_EMBEDDING_MODEL
    raise RuntimeError('No embedding credentials in .env')


def batched(iterable: Iterable, n: int) -> Iterator[list]:
    bucket = []
    for item in iterable:
        bucket.append(item)
        if len(bucket) >= n:
            yield bucket
            bucket = []
    if bucket:
        yield bucket


embed_client, EMBED_MODEL_NAME = build_embedding_client()


def embed_texts(texts: List[str], max_retries: int = 5) -> List[List[float]]:
    out: List[List[float]] = []
    for chunk in batched(texts, EMBED_BATCH):
        attempt = 0
        while True:
            try:
                resp = embed_client.embeddings.create(model=EMBED_MODEL_NAME, input=chunk)
                out.extend([d.embedding for d in resp.data])
                break
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    raise
                wait = min(2 ** attempt, 30)
                print(f'  embed retry {attempt} after {wait}s: {e}')
                time.sleep(wait)
    return out


driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session(database=NEO4J_DATABASE) as s:
    print('Neo4j ok:', s.run('RETURN 1 AS ok').single()['ok'])
print('Embedding model:', EMBED_MODEL_NAME)

In [ ]:
# --- Load retriever definitions from prod notebook (query-helpers cell) ----

import json as _json

assert PROD_NOTEBOOK.exists(), f'Missing prod notebook: {PROD_NOTEBOOK}'
_prod_nb = _json.loads(PROD_NOTEBOOK.read_text(encoding='utf-8'))
_helpers_cell = next(c for c in _prod_nb['cells'] if c.get('id') == 'query-helpers')
_helpers_src = ''.join(_helpers_cell['source'])
# Use same dict for globals and locals so defs land in the notebook namespace.
_g = globals()
exec(_helpers_src, _g, _g)

_REQUIRED = [
    'top_creators', 'count_creators_by_topic', 'unified_search', 'embed_query',
]
_missing = [n for n in _REQUIRED if not callable(_g.get(n))]
if _missing:
    raise RuntimeError(f'query-helpers did not define: {_missing}')

print('Loaded retrievers from query-helpers:', PROD_NOTEBOOK.name)
print('Sample:', ', '.join(_REQUIRED[:4]), '...')

In [ ]:
# --- CSV export helpers ------------------------------------------------------

def _cell_value(value: Any) -> Any:
    if value is None:
        return ''
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value


def _flatten_record(theme: str, record: Dict[str, Any], extra: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    row: Dict[str, Any] = {'theme': theme}
    if extra:
        row.update(extra)
    for key, value in record.items():
        row[key] = _cell_value(value)
    return row


def result_to_rows(theme: str, result: Any) -> List[Dict[str, Any]]:
    """Normalize list, count dict, or unified_search dict into flat CSV rows."""
    if isinstance(result, list):
        return [_flatten_record(theme, dict(item)) for item in result]

    if not isinstance(result, dict):
        return [{'theme': theme, 'error': f'unexpected result type: {type(result).__name__}'}]

    if 'videos' in result or 'creators' in result:
        rows: List[Dict[str, Any]] = []
        for video in result.get('videos') or []:
            rows.append(_flatten_record(theme, dict(video), {'record_type': 'video'}))
        for creator in result.get('creators') or []:
            rows.append(_flatten_record(theme, dict(creator), {'record_type': 'creator'}))
        if not rows:
            rows.append({'theme': theme, 'record_type': 'empty', 'note': 'no unified hits'})
        return rows

    # count_* payloads (creators or videos)
    return [_flatten_record(theme, result)]


def write_csv(path: Path, rows: List[Dict[str, Any]]) -> None:
    fieldnames: List[str] = []
    seen = set()
    for row in rows:
        for key in row.keys():
            if key not in seen:
                seen.add(key)
                fieldnames.append(key)
    if 'theme' in fieldnames:
        fieldnames = ['theme'] + [k for k in fieldnames if k != 'theme']
    else:
        fieldnames = ['theme'] + fieldnames

    with path.open('w', newline='', encoding='utf-8') as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        for row in rows:
            writer.writerow({k: row.get(k, '') for k in fieldnames})


def run_retriever(name: str, fn: Callable[[str], Any], themes: List[str]) -> Path:
    all_rows: List[Dict[str, Any]] = []
    for theme in themes:
        print(f'  {name} | {theme!r} ...', end=' ', flush=True)
        try:
            result = fn(theme)
            rows = result_to_rows(theme, result)
            all_rows.extend(rows)
            print(len(rows), 'row(s)')
        except Exception as exc:
            print('ERROR')
            all_rows.append({
                'theme': theme,
                'error': str(exc),
                'traceback': traceback.format_exc(),
            })
    out_path = OUTPUT_DIR / f'{name}.csv'
    write_csv(out_path, all_rows)
    return out_path

In [ ]:
# --- Register all retrievers (run after "Load retrievers" cell) ---------------

_kw = dict(top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE)
_count_kw = dict(min_score=MIN_SCORE, k=K_COUNT)


def _bind_retriever(name: str, **kwargs) -> Callable[[str], Any]:
    """Resolve retriever by name from globals() so this cell is safe after exec load."""
    fn = globals().get(name)
    if not callable(fn):
        raise RuntimeError(
            f"Retriever {name!r} is not defined. Run the 'Load retrievers' cell above first."
        )
    return lambda theme, _fn=fn, _kwargs=kwargs: _fn(theme, **_kwargs)


RETRIEVERS: List[Tuple[str, Callable[[str], Any]]] = [
    ('top_creators', _bind_retriever('top_creators', **_kw)),
    ('count_creators_by_topic', _bind_retriever('count_creators_by_topic', **_count_kw)),
    ('count_creators_by_content', _bind_retriever('count_creators_by_content', **_count_kw)),
    ('count_videos_by_topic', _bind_retriever('count_videos_by_topic', **_count_kw)),
    ('count_videos_by_comment_summary', _bind_retriever('count_videos_by_comment_summary', **_count_kw)),
    ('count_videos_by_content', _bind_retriever('count_videos_by_content', **_count_kw)),
    ('top_videos_by_topic_engagement', _bind_retriever('top_videos_by_topic_engagement', **_kw)),
    ('top_creators_by_topic_engagement', _bind_retriever('top_creators_by_topic_engagement', **_kw)),
    ('example_videos', _bind_retriever('example_videos', **_kw)),
    ('comment_discussions', _bind_retriever('comment_discussions', **_kw)),
    ('content_videos', _bind_retriever('content_videos', **_kw)),
    ('content_top_creators', _bind_retriever('content_top_creators', **_kw)),
    ('unified_search', _bind_retriever('unified_search', **_kw)),
]

print(f'Retrievers to export: {len(RETRIEVERS)}')

In [ ]:
# --- Run batch export --------------------------------------------------------

started = datetime.now(timezone.utc).isoformat()
written: List[Path] = []

print(f'Export started {started}')
for name, fn in RETRIEVERS:
    print(f'\n=== {name} ===')
    path = run_retriever(name, fn, TOPICS)
    written.append(path)
    print(f'  -> {path}')

manifest_path = OUTPUT_DIR / '_manifest.json'
manifest = {
    'started_at': started,
    'finished_at': datetime.now(timezone.utc).isoformat(),
    'topics': TOPICS,
    'top_n': TOP_N,
    'min_score': MIN_SCORE,
    'k': K_COUNT,
    'files': [p.name for p in written],
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(f'\nDone. {len(written)} CSV files + manifest in {OUTPUT_DIR.resolve()}')

In [ ]:
# Optional: close Neo4j driver when finished
try:
    driver.close()
except Exception:
    pass
print('Driver closed.')